# 03 — MobileNetV2 Fine-Tuning

**EN3150 Assignment 03 — Resource-Constrained CNN for Edge Image Classification**  
**Owner:** Rajitha  
**Environment:** Google Colab + TensorFlow/Keras

## Goal

Use **ImageNet-pretrained MobileNetV2** on the **same 17-class UCI Jute Pest split** used by the custom CNNs.

This notebook reuses Rajitha's Jute Pest cache from `01_model_a_standard.ipynb`, trains the MobileNetV2 classifier head, fine-tunes the final backbone layers, evaluates the model, and exports `mobilenetv2.json`.

### Relation to Rajitha's first notebook

Rajitha's Model A is intentionally a **higher-capacity custom Standard CNN** with several Conv2D blocks and Dense layers. MobileNetV2 should remain a **lightweight pretrained edge model** rather than being made similarly large. The useful comparison is whether transfer learning gives a better accuracy/memory/computation trade-off than the complex custom Model A.

In [ ]:
%pip install -q scikit-learn pillow

In [ ]:
import os
import gc
import time
import json
import random
import shutil
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
SEED = 42
IMG_SIZE = 64
BATCH_SIZE = 64

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("SEED =", SEED)
print("IMG_SIZE =", IMG_SIZE)
print("BATCH_SIZE =", BATCH_SIZE)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MEMBER = "rajitha"

# Same personal Jute Pest dataset cache used by Rajitha's Model A notebook.
PERSONAL_ROOT = Path(
    f"/content/drive/MyDrive/EN3150_A03_PERSONAL/{MEMBER}"
)

DATA_ROOT = (
    PERSONAL_ROOT
    / "dataset_cache"
    / "jute_pest"
)

# Separate artifacts/plots for MobileNetV2.
ARTIFACT_ROOT = (
    PERSONAL_ROOT
    / "artifacts"
    / "jute_pest"
    / "mobilenetv2"
)

PLOT_ROOT = (
    PERSONAL_ROOT
    / "plots"
    / "jute_pest"
    / "mobilenetv2"
)

for p in [DATA_ROOT, ARTIFACT_ROOT, PLOT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Shared group folder.
SHARED_ROOT = Path(
    "/content/drive/MyDrive/EN3150_A03_SHARED"
)

if not SHARED_ROOT.exists():
    raise FileNotFoundError(
        "EN3150_A03_SHARED was not found in My Drive. "
        "Add the shared folder as a shortcut directly inside My Drive."
    )

RESULT_ROOT = (
    SHARED_ROOT
    / "shared_results_jute_pest"
)

RESULT_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = (
    RESULT_ROOT
    / "jute_pest_split_manifest.csv"
)

SUMMARY_PATH = (
    RESULT_ROOT
    / "dataset_summary.json"
)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "jute_pest_split_manifest.csv is missing. "
        "Run Sahanya's shared dataset notebook first."
    )

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "dataset_summary.json is missing. "
        "Run Sahanya's shared dataset notebook first."
    )

print("Personal root:", PERSONAL_ROOT)
print("Dataset cache:", DATA_ROOT)
print("MobileNet artifacts:", ARTIFACT_ROOT)
print("Shared results:", RESULT_ROOT)

In [ ]:
# ============================================================
# UCI JUTE PEST DATASET — EXACT SHARED SPLIT
# ============================================================

DATASET_URL = (
    "https://archive.ics.uci.edu/static/public/920/"
    "jute+pest+dataset.zip"
)

ZIP_PATH = DATA_ROOT / "jute_pest_dataset.zip"
EXTRACT_DIR = DATA_ROOT / "extracted"

# ------------------------------------------------------------
# 1. Download only if Rajitha's Model A has not cached it already
# ------------------------------------------------------------
if not ZIP_PATH.exists():

    print("Downloading UCI Jute Pest dataset...")

    temp_zip = ZIP_PATH.with_suffix(".part")

    if temp_zip.exists():
        temp_zip.unlink()

    urllib.request.urlretrieve(
        DATASET_URL,
        temp_zip
    )

    temp_zip.replace(ZIP_PATH)

    print("Download complete.")

else:

    print("Using cached Jute Pest ZIP:", ZIP_PATH)


if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError(
        "The cached UCI file is not a valid ZIP."
    )


# ------------------------------------------------------------
# 2. Extract outer ZIP once
# ------------------------------------------------------------
EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

outer_marker = (
    EXTRACT_DIR
    / ".outer_extraction_complete"
)

if not outer_marker.exists():

    print("Extracting outer UCI archive...")

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as zf:

        zf.extractall(EXTRACT_DIR)

    outer_marker.write_text(
        "Outer ZIP extracted",
        encoding="utf-8"
    )

else:

    print("Outer archive already extracted.")


# ------------------------------------------------------------
# 3. Extract nested ZIPs recursively
# ------------------------------------------------------------
def extract_nested_zips(root_folder):

    processed = set()

    while True:

        nested_zips = [
            p
            for p in root_folder.rglob("*.zip")
            if str(p.resolve()) not in processed
        ]

        if not nested_zips:
            break

        for nested_zip in nested_zips:

            processed.add(
                str(nested_zip.resolve())
            )

            target_folder = (
                nested_zip.parent
                / f"{nested_zip.stem}_extracted"
            )

            marker = (
                target_folder
                / ".extraction_complete"
            )

            if marker.exists():
                continue

            print(
                "Extracting nested ZIP:",
                nested_zip.name
            )

            target_folder.mkdir(
                parents=True,
                exist_ok=True
            )

            if not zipfile.is_zipfile(nested_zip):
                continue

            with zipfile.ZipFile(
                nested_zip,
                "r"
            ) as zf:

                zf.extractall(target_folder)

            marker.write_text(
                "Nested ZIP extracted",
                encoding="utf-8"
            )


extract_nested_zips(EXTRACT_DIR)


# ------------------------------------------------------------
# 4. Load shared metadata
# ------------------------------------------------------------
with open(
    SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    dataset_summary = json.load(f)


CLASS_NAMES = list(
    dataset_summary["classes"]
)

NUM_CLASSES = int(
    dataset_summary["num_classes"]
)

assert NUM_CLASSES == 17
assert len(CLASS_NAMES) == 17


# ------------------------------------------------------------
# 5. Load the exact 70/15/15 split manifest
# ------------------------------------------------------------
manifest = pd.read_csv(
    MANIFEST_PATH
)

required_columns = {
    "relative_path",
    "label",
    "class_name",
    "split",
}

if not required_columns.issubset(
    manifest.columns
):
    raise RuntimeError(
        "The shared split manifest has unexpected columns."
    )


def build_split(split_name):

    part = (
        manifest[
            manifest["split"] == split_name
        ]
        .copy()
        .reset_index(drop=True)
    )

    paths = np.asarray(
        [
            str(
                EXTRACT_DIR
                / rel_path
            )
            for rel_path
            in part["relative_path"].astype(str)
        ],
        dtype=str,
    )

    labels = (
        part["label"]
        .to_numpy(dtype=np.int32)
    )

    missing = [
        p
        for p in paths
        if not Path(p).exists()
    ]

    if missing:

        print(
            "Example missing path:",
            missing[0]
        )

        raise FileNotFoundError(
            f"{len(missing)} manifest images were not found. "
            "Check the Jute Pest extraction."
        )

    return paths, labels


train_paths, train_labels = build_split("train")
val_paths, val_labels = build_split("validation")
test_paths, test_labels = build_split("test")


print()
print("Dataset: UCI Jute Pest")
print("Classes:", NUM_CLASSES)
print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))


# ------------------------------------------------------------
# 6. Build raw TensorFlow datasets
# ------------------------------------------------------------
def decode_image(path, label):

    image_bytes = tf.io.read_file(path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False,
    )

    image.set_shape(
        [None, None, 3]
    )

    return image, label


raw_train = (
    tf.data.Dataset
    .from_tensor_slices(
        (train_paths, train_labels)
    )
    .map(
        decode_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
)

raw_val = (
    tf.data.Dataset
    .from_tensor_slices(
        (val_paths, val_labels)
    )
    .map(
        decode_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
)

raw_test = (
    tf.data.Dataset
    .from_tensor_slices(
        (test_paths, test_labels)
    )
    .map(
        decode_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
)


def count_examples(ds):
    return int(
        tf.data.experimental
        .cardinality(ds)
        .numpy()
    )


print()
print("TensorFlow cardinality")
print("----------------------")
print("Train:", count_examples(raw_train))
print("Validation:", count_examples(raw_val))
print("Test:", count_examples(raw_test))

assert set(np.unique(train_labels)) == set(range(NUM_CLASSES))
assert set(np.unique(val_labels)) == set(range(NUM_CLASSES))
assert set(np.unique(test_labels)) == set(range(NUM_CLASSES))

print()
print(
    "Exact shared UCI Jute Pest split loaded successfully."
)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def preprocess(image, label):

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE],
        antialias=True,
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label


# Cache before shuffle so training can reshuffle every epoch.
train_ds = (
    raw_train
    .map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .shuffle(
        4096,
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    raw_val
    .map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    raw_test
    .map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(
    "Train batches:",
    tf.data.experimental.cardinality(train_ds).numpy()
)
print(
    "Validation batches:",
    tf.data.experimental.cardinality(val_ds).numpy()
)
print(
    "Test batches:",
    tf.data.experimental.cardinality(test_ds).numpy()
)

## Build MobileNetV2

This is intentionally different from Rajitha's **complex custom Model A**.

```text
Model A
= higher-capacity custom Standard CNN
= several Conv2D blocks + Dense 512/256/128

MobileNetV2
= pretrained lightweight edge/mobile model
= parameter-efficient depthwise/inverted-residual design
```

The new classifier has **17 outputs** for the UCI Jute Pest classes.

In [ ]:
def build_mobile():

    inputs = keras.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3),
        name="image"
    )

    # Training-only augmentation for better generalization.
    x = layers.RandomFlip(
        "horizontal",
        seed=SEED,
        name="aug_flip"
    )(inputs)

    x = layers.RandomRotation(
        0.05,
        seed=SEED,
        name="aug_rotate"
    )(x)

    x = layers.RandomZoom(
        0.10,
        seed=SEED,
        name="aug_zoom"
    )(x)

    x = layers.RandomContrast(
        0.10,
        seed=SEED,
        name="aug_contrast"
    )(x)

    # MobileNetV2 expects approximately [-1, +1].
    x = layers.Rescaling(
        1.0 / 127.5,
        offset=-1.0,
        name="mobilenet_preprocess"
    )(x)

    base = keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )

    base.trainable = False

    x = base(
        x,
        training=False
    )

    x = layers.GlobalAveragePooling2D(
        name="global_average_pool"
    )(x)

    x = layers.Dropout(
        0.25,
        seed=SEED,
        name="classifier_dropout"
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        name="logits"
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="MobileNetV2_Jute_Pest"
    )

    return model, base.name


mobile, BASE_NAME = build_mobile()

mobile.summary()

print(
    "Total parameters:",
    f"{mobile.count_params():,}"
)

print(
    "Stage 1 trainable parameters:",
    f"{int(sum(np.prod(v.shape) for v in mobile.trainable_weights)):,}"
)

---
### ✅ Git commit checkpoint

Suggested commit:

```text
feat: update MobileNetV2 for UCI Jute Pest shared split
```

File:

```text
notebooks/03_mobilenetv2.ipynb
```

## Stage 1 — train classifier head

In [ ]:
LOSS_FN=keras.losses.SparseCategoricalCrossentropy(from_logits=True)
class PersistentEpochTimer(keras.callbacks.Callback):
    def __init__(self,csv_path): super().__init__(); self.csv_path=Path(csv_path); self.csv_path.parent.mkdir(parents=True,exist_ok=True)
    def on_epoch_begin(self,epoch,logs=None): self.start=time.perf_counter()
    def on_epoch_end(self,epoch,logs=None):
        row=pd.DataFrame([{'epoch':int(epoch),'seconds':float(time.perf_counter()-self.start)}])
        row.to_csv(self.csv_path,mode='a',header=not self.csv_path.exists(),index=False)
def read_log(run_name):
    p=ARTIFACT_ROOT/run_name/'training_log.csv'
    if not p.exists(): return pd.DataFrame()
    d=pd.read_csv(p)
    return d.drop_duplicates(subset=['epoch'],keep='last').sort_values('epoch') if 'epoch' in d else d
def average_epoch_time(run_name):
    p=ARTIFACT_ROOT/run_name/'epoch_times.csv'
    if not p.exists(): return np.nan
    d=pd.read_csv(p).drop_duplicates(subset=['epoch'],keep='last')
    return float(d.seconds.mean()) if len(d) else np.nan
def fit_resumable(model,run_name,optimizer,epochs):
    rd=ARTIFACT_ROOT/run_name; rd.mkdir(parents=True,exist_ok=True)
    final=rd/'final.keras'; best=rd/'best.keras'; backup=rd/'backup'; log=rd/'training_log.csv'; timing=rd/'epoch_times.csv'
    if final.exists(): print('Completed run found:',run_name); return keras.models.load_model(final)
    model.compile(optimizer=optimizer,loss=LOSS_FN,metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
    callbacks=[keras.callbacks.BackupAndRestore(backup_dir=str(backup),save_freq='epoch',delete_checkpoint=False),keras.callbacks.ModelCheckpoint(str(best),monitor='val_accuracy',mode='max',save_best_only=True,verbose=1),keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=3,min_lr=1e-7,verbose=1),keras.callbacks.CSVLogger(str(log),append=True),PersistentEpochTimer(timing)]
    model.fit(train_ds,validation_data=val_ds,epochs=epochs,callbacks=callbacks,verbose=1)
    model.save(final); return model
def reset_run(run_name):
    import shutil
    p=ARTIFACT_ROOT/run_name
    if p.exists(): shutil.rmtree(p)
def plot_history(run_name,prefix):
    d=read_log(run_name)
    plt.figure(figsize=(8,5)); plt.plot(d.epoch+1,d.loss,label='Train'); plt.plot(d.epoch+1,d.val_loss,label='Validation'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(prefix+' Loss'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/(prefix.lower().replace(' ','_')+'_loss.png'),dpi=180); plt.show()
    plt.figure(figsize=(8,5)); plt.plot(d.epoch+1,d.accuracy,label='Train'); plt.plot(d.epoch+1,d.val_accuracy,label='Validation'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title(prefix+' Accuracy'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/(prefix.lower().replace(' ','_')+'_accuracy.png'),dpi=180); plt.show()

In [ ]:
STAGE1_RUN = "mobilenetv2_jute_stage1"
STAGE1_EPOCHS = 5

mobile, BASE_NAME = build_mobile()

mobile = fit_resumable(
    mobile,
    STAGE1_RUN,
    keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    STAGE1_EPOCHS,
)

plot_history(
    STAGE1_RUN,
    "MobileNetV2 Jute Stage 1"
)

## Stage 2 — fine tune last 20 backbone layers

In [ ]:
def unfreeze_last(
    model,
    base_name,
    n=20
):

    base = model.get_layer(
        base_name
    )

    base.trainable = True

    for layer in base.layers[:-n]:
        layer.trainable = False

    # Keep BatchNorm layers frozen during transfer-learning fine-tuning.
    for layer in base.layers[-n:]:
        layer.trainable = not isinstance(
            layer,
            layers.BatchNormalization
        )

    return model


mobile = unfreeze_last(
    mobile,
    BASE_NAME,
    n=20
)

print(
    "Stage 2 trainable parameters:",
    f"{int(sum(np.prod(v.shape) for v in mobile.trainable_weights)):,}"
)

In [ ]:
STAGE2_RUN = "mobilenetv2_jute_stage2"
STAGE2_EPOCHS = 15

mobile = fit_resumable(
    mobile,
    STAGE2_RUN,
    keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    STAGE2_EPOCHS,
)

plot_history(
    STAGE2_RUN,
    "MobileNetV2 Jute Fine Tuning"
)

---
### ✅ Git commit checkpoint

Suggested commit:

```text
exp: train and fine tune MobileNetV2 on Jute Pest
```

File:

```text
notebooks/03_mobilenetv2.ipynb
```

## Test evaluation

In [ ]:
def get_true_labels(ds):

    return np.concatenate(
        [
            y.numpy()
            for _, y in ds
        ]
    )


Y_TEST = get_true_labels(
    test_ds
)


def model_file_size_mb(path):

    return (
        Path(path).stat().st_size
        / (1024 ** 2)
    )


def benchmark_inference_ms_per_image(
    model,
    ds,
    max_batches=10
):

    batches = []
    n_images = 0

    for i, (x, _) in enumerate(ds):

        if i >= max_batches:
            break

        batches.append(x)
        n_images += int(x.shape[0])

    if not batches:
        return np.nan

    # Warm-up
    _ = model(
        batches[0],
        training=False
    )

    start = time.perf_counter()

    for x in batches:
        _ = model(
            x,
            training=False
        )

    return (
        (time.perf_counter() - start)
        * 1000
        / n_images
    )


def evaluate_model(
    model,
    name,
    save_name
):

    logits = model.predict(
        test_ds,
        verbose=0
    )

    pred = np.argmax(
        logits,
        axis=1
    )

    out = {
        "accuracy": float(
            accuracy_score(
                Y_TEST,
                pred
            )
        ),
        "precision_macro": float(
            precision_score(
                Y_TEST,
                pred,
                average="macro",
                zero_division=0
            )
        ),
        "recall_macro": float(
            recall_score(
                Y_TEST,
                pred,
                average="macro",
                zero_division=0
            )
        ),
    }

    print(
        json.dumps(
            out,
            indent=2
        )
    )

    print()
    print(
        classification_report(
            Y_TEST,
            pred,
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        Y_TEST,
        pred
    )

    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_NAMES
    )

    disp.plot(
        ax=ax,
        xticks_rotation=90,
        colorbar=False
    )

    ax.set_title(
        name
        + " — Confusion Matrix"
    )

    plt.tight_layout()

    plt.savefig(
        PLOT_ROOT
        / save_name,
        dpi=180
    )

    plt.show()

    return out

In [ ]:
best_model_path = (
    ARTIFACT_ROOT
    / STAGE2_RUN
    / "best.keras"
)

if not best_model_path.exists():
    raise FileNotFoundError(
        "Stage 2 best.keras was not found. "
        "Complete fine-tuning first."
    )

best = keras.models.load_model(
    best_model_path
)

metrics = evaluate_model(
    best,
    "MobileNetV2 — UCI Jute Pest",
    "mobilenetv2_jute_confusion_matrix.png"
)

log = read_log(
    STAGE2_RUN
)

result = {
    "dataset": "UCI Jute Pest",
    "num_classes": int(NUM_CLASSES),
    "model": "MobileNetV2",
    "owner": "Rajitha",
    "training_method": (
        "ImageNet transfer learning + final-layer fine-tuning"
    ),
    "optimizer": (
        "Adam: lr=0.001 head training, lr=0.00001 fine-tuning"
    ),
    "stage1_epochs": int(STAGE1_EPOCHS),
    "stage2_epochs": int(STAGE2_EPOCHS),
    "parameters": int(
        best.count_params()
    ),
    "trainable_parameters": int(
        sum(
            np.prod(v.shape)
            for v in best.trainable_weights
        )
    ),
    "model_size_mb": float(
        model_file_size_mb(
            best_model_path
        )
    ),
    "estimated_fp32_weight_mb": float(
        best.count_params()
        * 4
        / (1024 ** 2)
    ),
    "best_val_accuracy": float(
        log["val_accuracy"].max()
    ),
    "accuracy": metrics["accuracy"],
    "precision_macro": metrics["precision_macro"],
    "recall_macro": metrics["recall_macro"],
    "avg_epoch_time_s": float(
        average_epoch_time(
            STAGE2_RUN
        )
    ),
    "inference_ms_per_image": float(
        benchmark_inference_ms_per_image(
            best,
            test_ds
        )
    ),
}

result_file = (
    RESULT_ROOT
    / "mobilenetv2.json"
)

with open(
    result_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        indent=2
    )

print(
    json.dumps(
        result,
        indent=2
    )
)

print(
    "Shared result saved to:",
    result_file
)

## Interpretation

Discuss:

- frozen-head training vs fine-tuning,
- best validation accuracy,
- measured test accuracy,
- macro precision and macro recall,
- major class confusions,
- parameter count and saved model size,
- training/inference cost,
- the accuracy-versus-memory trade-off.

### Important comparison with Rajitha's first notebook

Rajitha's **Model A is intentionally a complex, higher-capacity Standard CNN** using multiple Conv2D blocks and Dense layers. MobileNetV2 should be interpreted as the pretrained lightweight alternative.

Do not assume that either model is automatically better. Compare their **measured** accuracy, model size, parameter count, epoch time and inference time.

For the final assignment discussion, Sahanya should still focus especially on the required comparison:

```text
Model B vs MobileNetV2 vs EfficientNetB0
```

because the main project objective is resource-constrained edge classification.

---
### ✅ Git commit checkpoint

Suggested commit:

```text
analysis: add MobileNetV2 Jute Pest evaluation results
```

File:

```text
notebooks/03_mobilenetv2.ipynb
```